# S04E01 — OKO Editor

Zadanie: wprowadzenie zmian w Centrum Operacyjnym OKO **wyłącznie przez API** (nie przez interfejs webowy).

Co musimy zrobić:
1. Zmienić klasyfikację raportu o mieście **Skolwin** z pojazdów/ludzi → **zwierzęta**
2. Znaleźć zadanie związane ze Skolwinem i oznaczyć je jako **wykonane** (z notatką o bobrach)
3. Dodać nowy incydent dla miasta **Komarowo** (ruch ludzi)
4. Wywołać akcję `done`

Panel webowy (tylko do czytania): https://oko.ag3nts.org/ (Zofia / Zofia2026!)

Podejście agentowe: Claude z narzędziami do komunikacji z API OKO.

In [1]:
import os, json, requests
from dotenv import load_dotenv
import anthropic

load_dotenv("../.env")
API_KEY = os.getenv("AI_DEVS_API_KEY")
ANTHROPIC_KEY = os.getenv("ANTHROPIC_API_KEY")

VERIFY_URL = "https://hub.ag3nts.org/verify"
TASK = "okoeditor"

client = anthropic.Anthropic(api_key=ANTHROPIC_KEY)

In [2]:
# Funkcja pomocnicza — wywołanie API OKO
def oko_api(answer: dict) -> dict:
    payload = {"apikey": API_KEY, "task": TASK, "answer": answer}
    r = requests.post(VERIFY_URL, json=payload)
    result = r.json()
    print(f">> OKO API: {json.dumps(answer)[:120]}")
    print(f"<< Odpowiedź: {json.dumps(result)[:200]}")
    return result

# Sprawdź dostępne akcje
oko_api({"action": "help"})

>> OKO API: {"action": "help"}
<< Odpowiedź: {"code": 120, "message": "OKO Editor API help.", "status": "success", "command": "help", "description": "Available commands and request syntax for okoeditor API.", "commands": [{"action": "help", "syn


{'code': 120,
 'message': 'OKO Editor API help.',
 'status': 'success',
 'command': 'help',
 'description': 'Available commands and request syntax for okoeditor API.',
 'commands': [{'action': 'help',
   'syntax': {'apikey': 'YOUR_API_KEY',
    'task': 'okoeditor',
    'answer': {'action': 'help'}},
   'notes': ['Returns this help message.',
    'No additional fields are required.']},
  {'action': 'update',
   'syntax': {'apikey': 'YOUR_API_KEY',
    'task': 'okoeditor',
    'answer': {'page': 'incydenty|notatki|zadania',
     'id': '32-char-hex-id',
     'action': 'update',
     'content': 'new description text (optional)',
     'title': 'new title (optional)',
     'done': 'YES|NO (only for page zadania, optional)'}},
   'required_fields': ['page', 'id', 'action'],
   'optional_fields': ['content', 'title', 'done'],
   'rules': ['At least one of "content" or "title" must be provided.',
    '"done" is allowed only for page "zadania".',
    'Page "uzytkownicy" is read-only and cannot b

## Definicja narzędzi agenta

In [3]:
TOOLS = [
    {
        "name": "oko_call",
        "description": (
            "Wywołuje API systemu OKO w centrum operacyjnym. "
            "Przekaż action i opcjonalne parametry jako obiekt JSON. "
            "Przykładowe akcje: help, listReports, getReport, updateReport, "
            "listTasks, completeTask, createIncident, done."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "answer": {
                    "type": "object",
                    "description": "Obiekt answer wysyłany do API (musi zawierać pole 'action' oraz opcjonalne parametry)"
                }
            },
            "required": ["answer"]
        }
    }
]

def handle_tool(name: str, input_data: dict) -> str:
    if name == "oko_call":
        result = oko_api(input_data["answer"])
        return json.dumps(result, ensure_ascii=False)
    return f"Nieznane narzędzie: {name}"

## Agent — pętla ReAct

In [4]:
SYSTEM_PROMPT = """Jesteś agentem operacyjnym wykonującym zadanie w systemie OKO.

Masz dostęp do narzędzia oko_call, które pozwala komunikować się z API systemu OKO.

Twoja lista zadań (wykonaj je WSZYSTKIE po kolei):
1. Wywołaj help, żeby poznać dostępne akcje API.
2. Pobierz listę raportów (incydentów) i znajdź raport dotyczący miasta SKOLWIN.
3. Zmień klasyfikację raportu Skolwin tak, aby dotyczył zwierząt (nie pojazdów ani ludzi).
4. Pobierz listę zadań i znajdź zadanie związane ze Skolwinem.
5. Oznacz to zadanie jako wykonane — w treści wpisz, że widziano zwierzęta (np. bobry).
6. Dodaj nowy raport o incydencie w mieście KOMAROWO — ruch ludzi.
7. Gdy wszystko gotowe, wywołaj akcję 'done'.

WAŻNE:
- Zawsze zacznij od 'help', żeby poznać dokładne nazwy akcji i pól.
- Sprawdzaj odpowiedzi API i dostosowuj kolejne kroki.
- Nie wolno Ci używać interfejsu webowego — tylko API przez narzędzie oko_call.
- Jeśli w odpowiedzi pojawi się flaga (FLG:), wyświetl ją wyraźnie."""

messages = [{"role": "user", "content": "Wykonaj wszystkie wymagane operacje w systemie OKO i zdobądź flagę."}]

MAX_STEPS = 30
flag = None

for step in range(MAX_STEPS):
    print(f"\n{'='*60}")
    print(f"Krok {step + 1}")
    print('='*60)

    response = client.messages.create(
        model="claude-opus-4-6",
        max_tokens=4096,
        system=SYSTEM_PROMPT,
        tools=TOOLS,
        messages=messages,
    )

    # Dodaj odpowiedź asystenta do historii
    messages.append({"role": "assistant", "content": response.content})

    # Wyświetl tekst (jeśli jest)
    for block in response.content:
        if hasattr(block, 'text'):
            print(f"\n[Agent]: {block.text[:500]}")

    # Sprawdź powód zatrzymania
    if response.stop_reason == "end_turn":
        print("\nAgent zakończył pracę.")
        break

    # Obsłuż wywołania narzędzi
    if response.stop_reason == "tool_use":
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                print(f"\n[Narzędzie]: {block.name}({json.dumps(block.input)[:200]})")
                result = handle_tool(block.name, block.input)
                print(f"[Wynik]: {result[:300]}")

                # Szukaj flagi w wynikach
                if "FLG:" in result or "{FLG" in result:
                    flag = result

                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": result
                })

        messages.append({"role": "user", "content": tool_results})

        if flag:
            print(f"\n{'*'*60}")
            print(f"FLAGA ZNALEZIONA: {flag}")
            print('*'*60)
            break

print("\nAgent zakończył pracę.")


Krok 1

[Agent]: 

Zaczynam od kroku 1 — wywołuję `help`, aby poznać dostępne akcje API.

[Narzędzie]: oko_call({"answer": {"action": "help"}})
>> OKO API: {"action": "help"}
<< Odpowiedź: {"code": 120, "message": "OKO Editor API help.", "status": "success", "command": "help", "description": "Available commands and request syntax for okoeditor API.", "commands": [{"action": "help", "syn
[Wynik]: {"code": 120, "message": "OKO Editor API help.", "status": "success", "command": "help", "description": "Available commands and request syntax for okoeditor API.", "commands": [{"action": "help", "syntax": {"apikey": "YOUR_API_KEY", "task": "okoeditor", "answer": {"action": "help"}}, "notes": ["Retu

Krok 2

[Agent]: Widzę dostępne komendy: `help`, `update`, `done`. Nie widzę jednak `listReports`, `getReport`, `listTasks` itp. Muszę poszukać sposobu na pobranie list. Spróbuję kilku wariantów — może trzeba użyć innego podejścia.

[Narzędzie]: oko_call({"answer": {"action": "listReports"}})
>> O

In [ ]:
# Wyświetl flagę z ostatniej wiadomości
if flag:
    print(f"Flaga: {flag}")
else:
    # Sprawdź w całej historii
    for msg in messages:
        content = str(msg.get('content', ''))
        if 'FLG' in content:
            print(f"Flaga znaleziona w historii: {content[:500]}")
            break